# Modified MeTALine Pipeline

## 1. Data Origin & Experimental Context 
- czy znowu pisać skąd są dane i na czym polegał eksperyment?

## 2. MeTAline Pipeline Execution
To process the raw FASTQ reads into a taxonomic count matrix, we utilized **MeTAline**, an automated and standardized bioinformatics pipeline [Official Repository](https://github.com/Gabaldonlab/meTAline).

The MeTAline workflow automated the following steps:
* **Trimming** - `Trimmomatic` - Removal of adapters and low-quality sequence ends.
* **Host Depletion** - `Hisat2` - Mapping reads to the *R. robini* genome. Reads that do NOT map proceed further as the presumed microbiome.
* **Identyfikacja taksonomiczna** 
   * `Kraken2` - A highly sensitive k-mer-based method allows for taxonomic profiling.
   * `MetaPhlAn4` A highly senstitive marker-based method allows for taxonomic profiling.
* **Profilowanie funkcjonalne**
   * `HUMAnN` – Stratifies community functional profiles, indicating which species contribute to each pathway.

- czy dodac info o wersji programów oraz jakich baz uzyłem i skąd je pobrałem

### 2.1 Pipeline Execution Command
The pipeline was executed on the server using the following primary command:
```bash
for file in configs_48/*.json; do
  echo "="
  echo ">>> STARTING THE PIPELINE FOR: $file <<<"
  echo "="
  time docker run --rm \
  -v /media/raid:/media/raid \
  -v /media/array_a/jorzechowski/wyniki_meTAline:/media/raid/home/jorzechowski/wyniki_analizy \
  -w /media/raid/home/jorzechowski/snakemake_pipeline/metaline/meTAline \
  metaline \
  snakemake -s meTAline.smk --configfile "$file" --cores 20 -p --rerun-incomplete
  done
```

- czy komenda ma byc bardziej ogólna?

## 3. Modifications To The Pipeline
The MeTAline pipeline is built on Snakemake. To optimize server resource management and prevent storage saturation, several custom modifications were made directly to the pipeline's rule files (`.smk`). By marking specific outputs as `temp()` and commenting out unnecessary log files, this optimization reduced overall storage usage by approximately 40%.

Below is a detailed breakdown of the modifications made within the `metaline.smk` file and the `rules/` directory:

* **metaline.smk**
   * Modified the `rule all` to comment out outputs that were not needed for downstream analysis (e.g., `kraken2.txt`, `ext1`, `ext2`).
* **rules**
   - **trimming.smk** 
      - Marked all `protected()` output files as `temp()`.
   - **host_depletion.smk** 
      * Marked intermediate mapping files (`.bam` and `.bai`) as `temp()`.
   - **kmer_taxonomy.smk**  
      * Removed the generation of `kraken2.txt` 
      * In `rule extract_reads`, both outputs were marked as `temp()` 
      * Additionally, a shell command (`rm {params.intermediate1} {params.intermediate2}`) was appended to manually clear intermediate sequences immediately after processing.
   - **Biobakery.smk**  
      * Added `temp()` to the outputs generating `.bz2` and `.sam` files. 
      * In the HUMAnN section, the `combined_reads` output was marked as `temp()` 
 
## 4. Final Data Extraction 
Although the MeTALine pipeline generates multiple output folders and files, only the taxonomic classification results were required for statistical analysis. The final dataset, `kraken_family_matrix.csv` (used in the R analysis), was compiled directly from the `profiled.txt` files located in the `KRAKEN_ASSIGN` directory. 

- czy info dokładnie jak stworzyłem kraken_family_matrix.csv na podstawie pliku profiled.txt